In [2]:
import pandas as pd

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

In [3]:
transactions = [
    ["Bread", "Milk"],
    ["Bread", "Milk", "Butter"],
    ["Bread", "Butter"],
    ["Milk", "Butter"],
    ["Bread", "Milk", "Butter"]
]

In [4]:
encoder = TransactionEncoder()

encoded_data = encoder.fit_transform(transactions)

df = pd.DataFrame(
    encoded_data,
    columns=encoder.columns_
)

df

,Bread,Butter,Milk
0,True,False,True
1,True,True,True
2,True,True,False
3,False,True,True
4,True,True,True


In [5]:
frequent_itemsets = fpgrowth(
    df,
    min_support=0.4,
    use_colnames=True
)

frequent_itemsets.sort_values(
    by="support",
    ascending=False
)

,support,itemsets
0,0.8,frozenset({Milk})
1,0.8,frozenset({Bread})
2,0.8,frozenset({Butter})
3,0.6,"frozenset({Bread, Milk})"
4,0.6,"frozenset({Bread, Butter})"
6,0.6,"frozenset({Milk, Butter})"
5,0.4,"frozenset({Bread, Milk, Butter})"


In [6]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.6
)

rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].sort_values(
    by="lift",
    ascending=False
)

,antecedents,consequents,support,confidence,lift
0,frozenset({Bread}),frozenset({Milk}),0.6,0.750000,0.937500
1,frozenset({Milk}),frozenset({Bread}),0.6,0.750000,0.937500
2,frozenset({Bread}),frozenset({Butter}),0.6,0.750000,0.937500
3,frozenset({Butter}),frozenset({Bread}),0.6,0.750000,0.937500
8,frozenset({Butter}),frozenset({Milk}),0.6,0.750000,0.937500
7,frozenset({Milk}),frozenset({Butter}),0.6,0.750000,0.937500
4,"frozenset({Bread, Milk})",frozenset({Butter}),0.4,0.666667,0.833333
6,"frozenset({Milk, Butter})",frozenset({Bread}),0.4,0.666667,0.833333
5,"frozenset({Bread, Butter})",frozenset({Milk}),0.4,0.666667,0.833333


In [7]:
useful_rules = rules[
    (rules["confidence"] >= 0.6) &
    (rules["lift"] > 1)
]

useful_rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]]

,antecedents,consequents,support,confidence,lift


# FP-Growth (Frequent Pattern Growth)

## 1. What is FP-Growth?

FP-Growth is an unsupervised pattern-mining algorithm used to discover frequent itemsets in transactional data.

Unlike Apriori, it avoids candidate itemset generation by building a compressed data structure called an **FP-tree** and mining frequent patterns from it.

## 2. Transactions

```python
transactions = [
    ["Bread", "Milk"],
    ["Bread", "Milk", "Butter"],
    ["Bread", "Butter"],
    ["Milk", "Butter"],
    ["Bread", "Milk", "Butter"]
]
```

Assume the minimum support is `0.4` (40%). An itemset must appear in at least 2 of the 5 transactions to be frequent.

## 3. Working of FP-Growth

### Step 1: Count Item Frequencies

| Item | Count | Support |
|---|---:|---:|
| Bread | 4 | 0.8 |
| Milk | 4 | 0.8 |
| Butter | 4 | 0.8 |

All three items satisfy the minimum support threshold.

### Step 2: Sort Items by Frequency

Arrange items in descending order of frequency. If frequencies are equal, use a consistent order.

**Item frequencies:**
- Bread: 4
- Milk: 4
- Butter: 4

Since all items have equal frequency, we choose the order: **Bread → Milk → Butter**.

**Sorted transactions:**

| Original Transaction | Sorted Transaction |
|---|---|
| Bread, Milk | Bread → Milk |
| Bread, Milk, Butter | Bread → Milk → Butter |
| Bread, Butter | Bread → Butter |
| Milk, Butter | Milk → Butter |
| Bread, Milk, Butter | Bread → Milk → Butter |

These sorted transactions are inserted into the FP-tree to build shared prefixes and discover frequent patterns efficiently.
Arrange items in descending frequency order. Since all items have equal frequency, we choose the consistent order Bread, Milk, Butter.

### Step 3: Build the FP-Tree

Insert each transaction in the chosen order, sharing common prefixes and updating node counts.

```text
Root
├── Bread:4
│   ├── Milk:2
│   │   └── Butter:1
│   └── Butter:1
└── Milk:1
    └── Butter:1
```

Each node's count indicates how many transactions follow that path.

### Step 4: Mine the FP-Tree

For each item, construct its **conditional pattern base**, which contains the prefix paths associated with that item and their counts.

Then build a **conditional FP-tree** from those paths and recursively mine it to discover frequent combinations.

### Step 5: Frequent Itemsets

With minimum support of 0.4, the frequent itemsets are:

| Frequent itemset | Count | Support |
|---|---:|---:|
| {Bread} | 4 | 0.8 |
| {Milk} | 4 | 0.8 |
| {Butter} | 4 | 0.8 |
| {Bread, Milk} | 3 | 0.6 |
| {Bread, Butter} | 3 | 0.6 |
| {Milk, Butter} | 3 | 0.6 |
| {Bread, Milk, Butter} | 2 | 0.4 |

## 4. Apriori vs. FP-Growth

| Apriori | FP-Growth |
|---|---|
| Generates candidate itemsets | Avoids Apriori-style candidate generation |
| Repeatedly scans the database to count candidates | Typically uses two database scans to build the FP-tree |
| Can become slow when many candidates are generated | Often more efficient when candidate generation is expensive |

## 5. Key Takeaway

FP-Growth compresses transactions into an FP-tree and recursively mines conditional pattern bases and conditional FP-trees to discover frequent itemsets without generating candidates like Apriori.